In [17]:
from google.colab import files
uploaded = files.upload()

Saving top_100_spotify_artists.csv to top_100_spotify_artists (2).csv


In [18]:
import pandas as pd
df = pd.read_csv('top_100_spotify_artists.csv')
df.head()

,rank,artist,monthly_listeners,daily_change,peak_rank,peak_listeners,genre
0,1,Bruno Mars,135153284,-52411,1,151079821,Pop/R&B/Funk
1,2,The Weeknd,115270855,104649,1,126192069,R&B/Pop
2,3,Bad Bunny,109354219,-475700,2,123934765,Reggaeton/Latin Trap
3,4,Rihanna,107446720,150033,2,107446720,Pop/R&B
4,5,Taylor Swift,102250746,-69267,1,116229071,Pop/Country


In [19]:
print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nMissing values:', df.isnull().sum().sum())
print('\nData types:')
print(df.dtypes)
print('\nBasic Statistics:')
print(df.describe())

Shape: (100, 7)

Columns: ['rank', 'artist', 'monthly_listeners', 'daily_change', 'peak_rank', 'peak_listeners', 'genre']

Missing values: 0

Data types:
rank                  int64
artist               object
monthly_listeners     int64
daily_change          int64
peak_rank             int64
peak_listeners        int64
genre                object
dtype: object

Basic Statistics:
             rank  monthly_listeners   daily_change   peak_rank  \
count  100.000000       1.000000e+02     100.000000  100.000000   
mean    50.500000       5.941319e+07   74127.840000   30.400000   
std     29.011492       1.824720e+07  179109.542861   23.563368   
min      1.000000       4.243502e+07 -475700.000000    1.000000   
25%     25.750000       4.697274e+07  -12315.000000    6.750000   
50%     50.500000       5.246743e+07   48540.000000   26.000000   
75%     75.250000       6.503518e+07  127716.500000   50.000000   
max    100.000000       1.351533e+08  827294.000000   83.000000   

       peak_l

In [20]:
# Artists above median monthly listeners = Popular (1)
# Artists below median monthly listeners = Less Popular (0)

df['label'] = (df['monthly_listeners'] > df['monthly_listeners'].median()).astype(int)
print('Label distribution:')
print(df['label'].value_counts())
print(f'\nMedian monthly listeners: {df["monthly_listeners"].median():,.0f}')

Label distribution:
label
1    50
0    50
Name: count, dtype: int64

Median monthly listeners: 52,467,430


In [21]:
%pip install nltk
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [22]:
def preprocess_text(text):
    # Lowercase
    text = str(text).lower()
    # Remove special characters
    text = re.sub(r"[^a-zA-Z]", " ", text)
    # Tokenize
    words = text.split()
    # Remove stopwords
    words = [w for w in words if w not in stop_words]
    # Lemmatize
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)
df['text_combined'] = df['artist'] + " " + df['genre']
df['clean_text'] = df['text_combined'].apply(preprocess_text)
df[['artist', 'genre', 'clean_text']].head(10)

,artist,genre,clean_text
0,Bruno Mars,Pop/R&B/Funk,bruno mar pop r b funk
1,The Weeknd,R&B/Pop,weeknd r b pop
2,Bad Bunny,Reggaeton/Latin Trap,bad bunny reggaeton latin trap
3,Rihanna,Pop/R&B,rihanna pop r b
4,Taylor Swift,Pop/Country,taylor swift pop country
5,Justin Bieber,Pop,justin bieber pop
6,Lady Gaga,Pop/Dance,lady gaga pop dance
7,Coldplay,Rock/Alternative,coldplay rock alternative
8,Drake,Hip-Hop/Rap,drake hip hop rap
9,Billie Eilish,Indie Pop/Alternative,billie eilish indie pop alternative


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=1000)
X_text = tfidf.fit_transform(df['clean_text'])
print('TF-IDF matrix shape:', X_text.shape)

TF-IDF matrix shape: (100, 197)


In [24]:
from sklearn.preprocessing import StandardScaler
numeric_features = df[['monthly_listeners', 'daily_change', 'peak_rank', 'peak_listeners']]
scaler = StandardScaler()
X_num = scaler.fit_transform(numeric_features)
print('Numeric feature matrix shape:', X_num.shape)

Numeric feature matrix shape: (100, 4)


In [25]:
from scipy.sparse import hstack, csr_matrix
X = hstack([X_text, csr_matrix(X_num)])
y = df['label']
print('Combined feature matrix shape:', X.shape)

Combined feature matrix shape: (100, 201)


In [26]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

Train size: 80, Test size: 20


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier()
nb = MultinomialNB()
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
nb.fit(X_train_text, y_train_text)
print('All models trained')

All models trained


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print('Accuracy :', accuracy_score(y_test, y_pred))
    print('Precision:', precision_score(y_test, y_pred, zero_division=0))
    print('Recall   :', recall_score(y_test, y_pred, zero_division=0))
    print('F1 Score :', f1_score(y_test, y_pred, zero_division=0))
print('Logistic Regression')
evaluate(lr, X_test, y_test)
print('\nDecision Tree')
evaluate(dt, X_test, y_test)
print('\nNaive Bayes')
evaluate(nb, X_test_text, y_test_text)

Logistic Regression
Accuracy : 0.95
Precision: 1.0
Recall   : 0.9166666666666666
F1 Score : 0.9565217391304348

Decision Tree
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Naive Bayes
Accuracy : 0.5
Precision: 1.0
Recall   : 0.16666666666666666
F1 Score : 0.2857142857142857
